# Week 3 — Classic Data Warehousing with Bike Store
## Star Schema + OLAP Queries

### Notebook Overview
This notebook turns the **Bike Store** dataset into a simple classic data warehouse lesson.

What this notebook does:
- explains the business problem
- creates a **star schema**
- loads a small Bike Store warehouse into DuckDB
- runs **OLAP-style queries**
- connects SQL results to business insight

### Business Context
Assume we are analysts for a bike retailer. The business wants to answer questions such as:
- Which products sell best?
- Which stores generate the most revenue?
- How does sales performance change over time?
- Which brands and categories drive the business?

### DW Concepts in this Notebook
- **Fact table**: sales facts at the order-line grain
- **Dimension tables**: date, customer, store, product, staff
- **OLAP queries**: roll-up, drill-down, slice, dice, ranking


## 1) Setup
We use DuckDB as the analytical engine.

Why DuckDB here?
- lightweight and local
- perfect for teaching
- fast enough for warehouse-style queries


In [ ]:
%load_ext sql
%sql duckdb:///week03_bikestore_dw.duckdb

## 2) Create a Small Bike Store Star Schema

### Grain
The fact table grain is:

> **one row per order line item**

This means each row represents one product sold on one order line.

Why this grain matters:
- detailed enough for product analysis
- flexible enough for time, store, and customer analysis
- realistic for classic DW teaching


In [ ]:

%sql DROP TABLE IF EXISTS fact_sales
%sql DROP TABLE IF EXISTS dim_date
%sql DROP TABLE IF EXISTS dim_customer
%sql DROP TABLE IF EXISTS dim_store
%sql DROP TABLE IF EXISTS dim_product
%sql DROP TABLE IF EXISTS dim_staff


In [ ]:

%sql CREATE TABLE dim_date (
    date_id INTEGER,
    full_date DATE,
    year INTEGER,
    quarter INTEGER,
    month INTEGER,
    month_name VARCHAR
)


In [ ]:

%sql CREATE TABLE dim_customer (
    customer_id INTEGER,
    customer_name VARCHAR,
    city VARCHAR,
    state VARCHAR
)


In [ ]:

%sql CREATE TABLE dim_store (
    store_id INTEGER,
    store_name VARCHAR,
    city VARCHAR,
    state VARCHAR
)


In [ ]:

%sql CREATE TABLE dim_product (
    product_id INTEGER,
    product_name VARCHAR,
    brand_name VARCHAR,
    category_name VARCHAR,
    model_year INTEGER,
    list_price DOUBLE
)


In [ ]:

%sql CREATE TABLE dim_staff (
    staff_id INTEGER,
    staff_name VARCHAR
)


In [ ]:

%sql CREATE TABLE fact_sales (
    order_id INTEGER,
    line_no INTEGER,
    date_id INTEGER,
    customer_id INTEGER,
    store_id INTEGER,
    product_id INTEGER,
    staff_id INTEGER,
    quantity INTEGER,
    unit_price DOUBLE,
    discount DOUBLE,
    revenue DOUBLE
)


## 3) Load Sample Dimension Data

This sample data is intentionally small enough for teaching:
- easy to inspect
- easy to explain
- enough to support realistic DW questions


In [ ]:

%sql INSERT INTO dim_date VALUES
(20220115, '2022-01-15', 2022, 1, 1, 'Jan'),
(20220210, '2022-02-10', 2022, 1, 2, 'Feb'),
(20220305, '2022-03-05', 2022, 1, 3, 'Mar'),
(20220420, '2022-04-20', 2022, 2, 4, 'Apr'),
(20230112, '2023-01-12', 2023, 1, 1, 'Jan'),
(20230218, '2023-02-18', 2023, 1, 2, 'Feb'),
(20230322, '2023-03-22', 2023, 1, 3, 'Mar'),
(20230425, '2023-04-25', 2023, 2, 4, 'Apr')


In [ ]:

%sql INSERT INTO dim_customer VALUES
(1, 'Alice Johnson', 'San Jose', 'CA'),
(2, 'Bob Smith', 'Sunnyvale', 'CA'),
(3, 'Carla Gomez', 'Palo Alto', 'CA'),
(4, 'David Lee', 'Fremont', 'CA'),
(5, 'Emma Brown', 'Mountain View', 'CA')


In [ ]:

%sql INSERT INTO dim_store VALUES
(1, 'Downtown Bikes', 'San Jose', 'CA'),
(2, 'Valley Bikes', 'Sunnyvale', 'CA'),
(3, 'Peninsula Bikes', 'Palo Alto', 'CA')


In [ ]:

%sql INSERT INTO dim_product VALUES
(101, 'Road-100', 'Trek', 'Road Bikes', 2022, 1200),
(102, 'Road-200', 'Giant', 'Road Bikes', 2023, 1800),
(103, 'Mountain-100', 'Trek', 'Mountain Bikes', 2022, 1500),
(104, 'Helmet Pro', 'Bell', 'Accessories', 2023, 80),
(105, 'Bike Pump', 'Park Tool', 'Accessories', 2023, 45),
(106, 'Hybrid-300', 'Specialized', 'Hybrid Bikes', 2023, 950)


In [ ]:

%sql INSERT INTO dim_staff VALUES
(1, 'Mia Chen'),
(2, 'Noah Patel'),
(3, 'Olivia Garcia')


## 4) Load Sample Fact Data

The fact table contains measurable business activity:
- quantity
- unit price
- discount
- revenue

Revenue is precomputed here for simplicity:
> `revenue = quantity * unit_price * (1 - discount)`


In [ ]:

%sql INSERT INTO fact_sales VALUES
(1001, 1, 20220115, 1, 1, 101, 1, 1, 1200, 0.05, 1140),
(1001, 2, 20220115, 1, 1, 104, 1, 2, 80, 0.00, 160),
(1002, 1, 20220210, 2, 2, 103, 2, 1, 1500, 0.10, 1350),
(1003, 1, 20220305, 3, 3, 102, 3, 1, 1800, 0.00, 1800),
(1003, 2, 20220305, 3, 3, 105, 3, 1, 45, 0.00, 45),
(1004, 1, 20220420, 4, 1, 106, 1, 2, 950, 0.05, 1805),
(1005, 1, 20230112, 5, 2, 101, 2, 1, 1200, 0.00, 1200),
(1005, 2, 20230112, 5, 2, 104, 2, 1, 80, 0.00, 80),
(1006, 1, 20230218, 2, 1, 102, 1, 1, 1800, 0.15, 1530),
(1007, 1, 20230322, 1, 3, 103, 3, 1, 1500, 0.00, 1500),
(1008, 1, 20230425, 4, 2, 106, 2, 1, 950, 0.00, 950),
(1008, 2, 20230425, 4, 2, 105, 2, 2, 45, 0.00, 90)


## 5) Inspect the Star Schema

This step is pedagogically important:
- students should see the warehouse tables
- students should distinguish fact vs dimension
- students should connect schema design to later SQL


In [ ]:
%sql SELECT * FROM dim_product

In [ ]:
%sql SELECT * FROM dim_store

In [ ]:
%sql SELECT * FROM fact_sales

## 6) First Warehouse Query — Total Revenue

### Business Question
How much revenue does the business generate overall?

### Why this matters
This is the simplest KPI in the warehouse:
- baseline performance
- sanity check
- starting point for more detailed analysis


In [ ]:

res = %sql SELECT SUM(revenue) AS total_revenue FROM fact_sales
res.DataFrame()


## 7) Sales by Store

### Business Question
Which stores generate the most revenue?

### DW Idea
This query joins:
- fact table = measurable events
- store dimension = context

### Business Insight
This helps:
- compare store performance
- identify top-performing locations
- support resource allocation


In [ ]:

res = %sql
SELECT
    s.store_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_store s
  ON f.store_id = s.store_id
GROUP BY s.store_name
ORDER BY total_revenue DESC

store_df = res.DataFrame()
store_df


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(store_df["store_name"], store_df["total_revenue"])
plt.title("Revenue by Store")
plt.xlabel("Store")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 8) Sales by Product Category

### Business Question
Which categories drive the business?

### Why this matters
Category analysis supports:
- product strategy
- inventory planning
- merchandising decisions


In [ ]:

res = %sql
SELECT
    p.category_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p
  ON f.product_id = p.product_id
GROUP BY p.category_name
ORDER BY total_revenue DESC

cat_df = res.DataFrame()
cat_df


In [ ]:

plt.figure(figsize=(8, 4))
plt.bar(cat_df["category_name"], cat_df["total_revenue"])
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 9) Sales by Brand

### Business Question
Which brands contribute most to total revenue?

### Business Insight
Brand performance can influence:
- supplier relationships
- promotion decisions
- assortment planning


In [ ]:

res = %sql
SELECT
    p.brand_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p
  ON f.product_id = p.product_id
GROUP BY p.brand_name
ORDER BY total_revenue DESC

brand_df = res.DataFrame()
brand_df


In [ ]:

plt.figure(figsize=(8, 4))
plt.bar(brand_df["brand_name"], brand_df["total_revenue"])
plt.title("Revenue by Brand")
plt.xlabel("Brand")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 10) Time Analysis — Revenue by Year and Month

### Business Question
How does revenue change over time?

### DW Idea
Time is almost always a conformed dimension in warehousing.

### Business Insight
Time analysis supports:
- forecasting
- seasonality detection
- planning and budgeting


In [ ]:

res = %sql
SELECT
    d.year,
    d.month,
    d.month_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_date d
  ON f.date_id = d.date_id
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month

time_df = res.DataFrame()
time_df


In [ ]:

time_df["period"] = time_df["year"].astype(str) + "-" + time_df["month_name"]

plt.figure(figsize=(9, 4))
plt.plot(time_df["period"], time_df["total_revenue"], marker="o")
plt.title("Revenue by Month")
plt.xlabel("Period")
plt.ylabel("Revenue")
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11) Drill-Down Example

### OLAP Concept
A warehouse supports drill-down:
- from higher-level summary
- to more detailed views
- for investigation

### Business Example
Start with revenue by year, then drill down to month.


In [ ]:

res = %sql
SELECT
    d.year,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_date d
  ON f.date_id = d.date_id
GROUP BY d.year
ORDER BY d.year

year_df = res.DataFrame()
year_df


In [ ]:

plt.figure(figsize=(6, 4))
plt.bar(year_df["year"].astype(str), year_df["total_revenue"])
plt.title("Revenue by Year")
plt.xlabel("Year")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 12) Slice Example

### OLAP Concept
Slice means fixing one dimension value.

### Business Question
What are sales for only the California bike business segment represented here by store locations?

### Teaching Note
In this dataset, all stores are in CA, so the point is conceptual:
students should understand the SQL pattern.


In [ ]:

res = %sql
SELECT
    s.store_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_store s
  ON f.store_id = s.store_id
WHERE s.state = 'CA'
GROUP BY s.store_name
ORDER BY total_revenue DESC

res.DataFrame()


## 13) Dice Example

### OLAP Concept
Dice means filtering across multiple dimensions.

### Business Question
How did Accessories perform in 2023?

### Business Insight
This is closer to real management questioning:
- a category
- over a time period
- for targeted decision-making


In [ ]:

res = %sql
SELECT
    d.year,
    p.category_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_date d
  ON f.date_id = d.date_id
JOIN dim_product p
  ON f.product_id = p.product_id
WHERE d.year = 2023
  AND p.category_name = 'Accessories'
GROUP BY d.year, p.category_name

res.DataFrame()


## 14) Customer Analysis

### Business Question
Which customers generate the most revenue?

### Business Insight
Customer-level analysis supports:
- loyalty programs
- VIP identification
- retention strategy


In [ ]:

res = %sql
SELECT
    c.customer_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_customer c
  ON f.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY total_revenue DESC

cust_df = res.DataFrame()
cust_df


In [ ]:

plt.figure(figsize=(9, 4))
plt.bar(cust_df["customer_name"], cust_df["total_revenue"])
plt.title("Revenue by Customer")
plt.xlabel("Customer")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 15) Top Products per Store
This query bridges Week 2 and Week 3:

- Week 2 skill: window functions
- Week 3 skill: fact + dimensions
- Combined result: ranked warehouse analytics


In [ ]:

res = %sql
SELECT *
FROM (
    SELECT
        s.store_name,
        p.product_name,
        SUM(f.revenue) AS revenue,
        ROW_NUMBER() OVER (
            PARTITION BY s.store_name
            ORDER BY SUM(f.revenue) DESC
        ) AS rn
    FROM fact_sales f
    JOIN dim_store s
      ON f.store_id = s.store_id
    JOIN dim_product p
      ON f.product_id = p.product_id
    GROUP BY s.store_name, p.product_name
) t
WHERE rn <= 2
ORDER BY store_name, rn

top_prod_df = res.DataFrame()
top_prod_df


## 16) Staff Performance

### Business Question
Which staff members are associated with the most revenue?

### Business Insight
This kind of query can support:
- performance reviews
- incentive programs
- staffing decisions


In [ ]:

res = %sql
SELECT
    st.staff_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_staff st
  ON f.staff_id = st.staff_id
GROUP BY st.staff_name
ORDER BY total_revenue DESC

staff_df = res.DataFrame()
staff_df


In [ ]:

plt.figure(figsize=(7, 4))
plt.bar(staff_df["staff_name"], staff_df["total_revenue"])
plt.title("Revenue by Staff")
plt.xlabel("Staff")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 17) Average Order-Line Revenue by Category

### Business Question
What is the average revenue contribution per order line by category?

### Business Insight
This helps compare:
- big-ticket categories
- accessory categories
- per-line profitability characteristics


In [ ]:

res = %sql
SELECT
    p.category_name,
    AVG(f.revenue) AS avg_line_revenue
FROM fact_sales f
JOIN dim_product p
  ON f.product_id = p.product_id
GROUP BY p.category_name
ORDER BY avg_line_revenue DESC

avg_cat_df = res.DataFrame()
avg_cat_df


## 18) Revenue by Customer State and Store

### Business Question
How do customer geography and store geography interact?

### Teaching Goal
Even with a small dataset, this shows multi-dimensional thinking:
- customer dimension
- store dimension
- fact table in the middle


In [ ]:

res = %sql
SELECT
    c.state AS customer_state,
    s.store_name,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_customer c
  ON f.customer_id = c.customer_id
JOIN dim_store s
  ON f.store_id = s.store_id
GROUP BY c.state, s.store_name
ORDER BY total_revenue DESC

res.DataFrame()


## 19) OLAP Tutorial Summary

### Core OLAP operations seen here
- roll-up: month → year
- drill-down: year → month
- slice: one state
- dice: one year + one category

### Business Value
OLAP helps decision-makers move across levels of abstraction quickly.


## 20) Final Conceptual Summary

What students should now understand:
- a fact table stores measurable business events
- dimensions provide context for analysis
- a star schema makes analytical SQL simpler

Why this matters:
- easier reporting
- faster querying
- better business decisions
